<a href="https://colab.research.google.com/github/LucasPuertas95/Procesamiento-del-Habla/blob/main/Trabajo_Procesamineto_de_Archivo_PDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LUCAS PUERTAS
# Trabajo de Extraccion y Procesamiento de Textos con PDFs

1_ **Adquisicion de Metadatos**

Aqui importo el pdf a colab desde un enlace.
Para poder extraer los metadatos del PDF

In [ ]:
!pip install -q pymupdf

import urllib.request
import os
import fitz

# Aqui descargo el archivo, guardo en una variable la direccion y luego lo descargo usando urlretrieve de urlib
URL = "https://www.indec.gob.ar/ftp/cuadros/publicaciones/programa_trabajo_indec_2025_2026.pdf"
urllib.request.urlretrieve(URL, "programa_trabajo_indec_2025_2026.pdf")
documento_PDF = "programa_trabajo_indec_2025_2026.pdf"

if os.path.exists(documento_PDF):
  documento = fitz.open(documento_PDF)
  metadatos = documento.metadata

  print("\n--- METADATOS EXTRAÍDOS ---")
  print(f"Título: {metadatos.get('title')}")
  print(f"Autor: {metadatos.get('author')}")
  print(f"Páginas: {documento.page_count}")
  print(f"Software: {metadatos.get('creator')}")

  documento.close()
else:
  print(f"El archivo {documento_PDF} no existe.")





--- METADATOS EXTRAÍDOS ---
Título: Programa Anual de Trabajo 2025-2026
Autor: Instituto Nacional de Estadística y Censos (INDEC)
Páginas: 87
Software: Adobe InDesign 21.0 (Windows)


2_ **Analisis Estructural del Texto**

Trabajamos aqui para extraer todo el contenido crudo de cinco paginas del pdf, elegimos cinco pero podriamos extraer todas las paginas del pdf

In [ ]:

# nuestaras variables donde tenemos el archvio y la apertura
documento_PDF = "programa_trabajo_indec_2025_2026.pdf"
documento = fitz.open(documento_PDF)

print("--- EXTRACCIÓN CRUDA (PRIMERAS 5 PÁGINAS) ---\n")

# las primeras 5 páginas
for i in range(5):
    pagina = documento.load_page(i)

    # La función 'get_text' con el parámetro "text" extrae el contenido crudo
    texto_crudo = pagina.get_text("text")

    print(f"--- PÁGINA {i+1} ---")
    print(texto_crudo)
    print("\n" + "="*50 + "\n")

documento.close()

--- EXTRACCIÓN CRUDA (PRIMERAS 5 PÁGINAS) ---

--- PÁGINA 1 ---
República Argentina
Instituto Nacional de
Estadística y Censos
Programa Anual de Trabajo
 2025-2026
Instituto Nacional de Estadística y Censos
Documento de trabajo Nº 51
ISBN: 978-950-896-706-0



--- PÁGINA 2 ---
Instituto Nacional de Estadística y Censos (INDEC)
   Programa anual de trabajo 2025-2026  : documento de trabajo N° 51. - 1a ed. - Ciudad Autónoma de Buenos 
Aires : Instituto Nacional de Estadística y Censos - INDEC, 2025.
   Libro digital, PDF - (Documentos de trabajo ; 51)
   Archivo Digital: descarga y online
   ISBN 978-950-896-706-0
1. Estadísticas. 2. Sistemas de Planificación. 3. Encuestas.
CDD 352.75
Buenos Aires, diciembre de 2025
Programa Anual de Trabajo 2025-2026
Documento de trabajo N° 51
Instituto Nacional de Estadística y Censos (INDEC)
Dirección: Marco Lavagna 
Dirección Técnica: Pedro Ignacio Lines
Dirección Nacional de Planificación, Relaciones Institucionales e Internacionales: Pablo Ceballos

3_ **Extraccion de Datos Tabulares**

Aqui importo pdfplumber y Pandas para convertir una tabla en una pagina a datos tabulares en un data frame de pandas

In [ ]:
!pip install -q pdfplumber pandas

import pdfplumber
import pandas as pd

with pdfplumber.open("programa_trabajo_indec_2025_2026.pdf") as PDF:
    pagina = PDF.pages[52]

    # Extraigo la tabla (devuelve una lista de listas)
    tabla = pagina.extract_table()

# lo convierto a DataFrame de pandas
my_data_frame = pd.DataFrame(tabla)

# aqui uso la primera fila como encabezado
my_data_frame.columns = my_data_frame.iloc[0]
my_data_frame = my_data_frame[1:]

print("Tabla extraída y limpia:")
display(my_data_frame.head())

Tabla extraída y limpia:


,Producto,Indicadores demográficos
1,Objetivo,Integrar y difundir indicadores clave sobre la...
2,Temas,"Evolución poblacional, tasas de crecimiento in..."
3,Cobertura,Nacional y jurisdiccional.
4,Relevamiento\nde datos,Recolección de datos mediante censos de poblac...
5,Periodicidad,"Actualizaciones anuales o trimestrales, según ..."


4_ **Análisis de frecuencias (NLP Inicial)**

Primero la tokenizacion, uso la libreira NLTK ya que es mas eficiente que SpaCy en trabajos pequeños y sencillos. SpaCy es mas compleja y literaria  

Luego trabajo usando stop words para filtrar las palabras no relevantes y repetitivas.



In [ ]:
import nltk
import plotly.express as px
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
from collections import Counter

nltk.download('punkt_tab')
nltk.download('stopwords')

# tokenizacion de la pagina elegida con word_tokenize de NLTK
documento = fitz.open(documento_PDF)
pagina = documento[9].get_text("text")
tokens = word_tokenize(pagina.lower(), language = "spanish")
documento.close()


# aqui ya al estar tokenizado aplico stop words
stop_words = set(stopwords.words('spanish'))
print("------STOP WORDS------")
print(stop_words)
print("----------------------")

# aqui filtro los tokens para eliminar las stop words
tokens_filtrados = [word for word in tokens if word.isalpha() and word not in stop_words]

conteo = Counter(tokens_filtrados).most_common(15)

# ahroa el conteo se convierte en un DataFrame para Plotly Express
df_frecuencias = pd.DataFrame(conteo, columns=['Palabra', 'Frecuencia'])

visualizacion = px.bar(df_frecuencias,
             x='Frecuencia',
             y='Palabra',
             orientation='h',
             title='Top 15 Palabras más frecuentes - Página Elegida (INDEC)',
             labels={'Frecuencia': 'Número de apariciones', 'Palabra': 'Tokens'},
             color='Frecuencia',
             color_continuous_scale='Viridis') # Una paleta de colores linda y profesional

# Ajustamos el diseño para que las palabras del eje Y no se corten
visualizacion.update_layout(yaxis={'categoryorder':'total ascending'}) # Ordena de mayor a menor frecuencia

visualizacion.show()

------STOP WORDS------
{'hasta', 'sois', 'hubiéramos', 'mía', 'sentidos', 'y', 'e', 'muy', 'tenías', 'vosotras', 'les', 'más', 'seáis', 'estadas', 'otros', 'estamos', 'tuyos', 'vuestras', 'fuéramos', 'habremos', 'tuviste', 'hubisteis', 'estás', 'fueseis', 'ellos', 'un', 'estemos', 'tenido', 'estuvieron', 'eso', 'soy', 'contra', 'estoy', 'sería', 'tendrás', 'para', 'seréis', 'nosotros', 'hubiésemos', 'tened', 'seas', 'ti', 'estuvieses', 'le', 'esos', 'tenían', 'seremos', 'cuando', 'tuvieran', 'hubiera', 'habrías', 'estando', 'nuestra', 'estarás', 'mis', 'fui', 'míos', 'habríais', 'hubieseis', 'tendría', 'estos', 'suyos', 'tendrá', 'éramos', 'hubiesen', 'durante', 'vuestra', 'tenidas', 'siente', 'sentid', 'tiene', 'fueras', 'habidos', 'tendréis', 'nos', 'esas', 'sentidas', 'tienes', 'fue', 'estuviera', 'tuve', 'estará', 'estarías', 'han', 'estaremos', 'estaríais', 'tendré', 'entre', 'tuyo', 'estaré', 'tuviera', 'hube', 'ellas', 'fuese', 'tuviésemos', 'teniendo', 'uno', 'suyas', 'eran', '

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
